### **2.fraud data preprocessing**

### 2.1 introduction

This notebook focuses on the preparing of the IEEE-CIS fraud deduction data set for machine learning and the deep learning models. The data set contains the numerical and the categorical features missing values, and an imbalance target variable.

The preprocessing process will include inspecting the dataset, splitting of the data set according to the transaction time, handling of the missing values and preparing features for the model training.

And the main purpose is to prepare reliable data for fraud detection while also preventing data leakage. A chronological split will be used so that the models are trained on the earlier transaction and evaluated on the later transactions.


In [13]:

# Install KaggleHub to access the IEEE-CIS Fraud Detection dataset
!pip -q install kagglehub

# Import the libraries required for preprocessing
import os
import gc
import numpy as np
import pandas as pd
import kagglehub

print("Libraries imported successfully!")


Libraries imported successfully!


In [14]:

# Download the IEEE-CIS Fraud Detection dataset from Kaggle
dataset_path = kagglehub.competition_download(
    "ieee-fraud-detection"
)

print("Dataset downloaded successfully!")
print("Dataset location:", dataset_path)
print("Available files:", os.listdir(dataset_path))


Dataset downloaded successfully!
Dataset location: /root/.cache/kagglehub/competitions/ieee-fraud-detection
Available files: ['train_identity.csv', 'train_transaction.csv', 'test_transaction.csv', 'test_identity.csv', 'sample_submission.csv']


In [15]:

# Load the IEEE-CIS training transaction dataset
file_path = os.path.join(
    dataset_path,
    "train_transaction.csv"
)

# Read the dataset
df = pd.read_csv(file_path, low_memory=False)

# Reduce memory usage for numerical columns
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

for col in df.select_dtypes(include=["int64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print(
    "Memory usage (MB):",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2)
)

display(df.head())


Dataset loaded successfully!
Dataset shape: (590540, 394)
Memory usage (MB): 1203.22


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



### 2.2 Data inspection

Before processing the dataset, it is expected to confirm its structure, identifying the numerical in the categorical features, checking the target variable as well as examine the missing values. This helps to determine in which preprocessing techniques are suitable for fraud detection models. The data set contains 590, 540 transactions and 394 columns. The target variable is fraud with the 0 representing the legitimate transaction and one representing a fraudulent transaction.


In [16]:

# Section 2.2: Dataset Inspection

print("DATASET INSPECTION")
print("-" * 45)

# Dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

# Identify numerical and categorical features
numerical_cols = df.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

# Check the target variable
print("\nFraud distribution:")
print(df["isFraud"].value_counts())

# Check missing values
missing_cols = df.isnull().sum()
print("\nColumns containing missing values:",
      (missing_cols > 0).sum())

# Check duplicated transaction IDs
print("\nDuplicate TransactionID values:",
      df["TransactionID"].duplicated().sum())


DATASET INSPECTION
---------------------------------------------
Number of rows: 590540
Number of columns: 394

Numerical columns: 380
Categorical columns: 14

Fraud distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64

Columns containing missing values: 374

Duplicate TransactionID values: 0


### 2.3 Temporal data splitting


The data said it is divided into training (70%), validation (15%) and the testing (15%) SIDS using the transactionsDT column. Unlike the random splitting, temporal splitting it keeps transaction in chronological
The earliest transactions are used for the training and the following transaction for the validation and the latest transaction for the testing. This approach reduces the risk of the use of future information during model development and supports the evaluation of the model performance over time.
All the preprocessing operations that are learned from the data such as importation and the feature encoding would be fitted using the training said only.



In [17]:

# Section 2.3: Temporal Data Splitting

# Sort transactions chronologically
df = df.sort_values("TransactionDT").reset_index(drop=True)

# Define split positions
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Create training, validation and testing sets
train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("TEMPORAL DATA SPLITTING")
print("-" * 45)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Testing set:", test_df.shape)

# Check fraud distribution in each split
for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    fraud_count = data["isFraud"].sum()
    fraud_rate = data["isFraud"].mean() * 100

    print(f"\n{name} set:")
    print("Fraud transactions:", fraud_count)
    print(f"Fraud rate: {fraud_rate:.2f}%")

# Confirm chronological order
assert train_df["TransactionDT"].max() <= val_df["TransactionDT"].min()
assert val_df["TransactionDT"].max() <= test_df["TransactionDT"].min()

print("\nChronological order verified!")


TEMPORAL DATA SPLITTING
---------------------------------------------
Training set: (413378, 394)
Validation set: (88581, 394)
Testing set: (88581, 394)

Training set:
Fraud transactions: 14538
Fraud rate: 3.52%

Validation set:
Fraud transactions: 3042
Fraud rate: 3.43%

Testing set:
Fraud transactions: 3083
Fraud rate: 3.48%

Chronological order verified!


### 2.4 Handling missing values

The data said it contains several features so what's the meeting values, which need to be addressed before the model training. And the missing value can affect the performance of machine learning and the deep learning models if they are not handled properly.

To prevent the leakage of the data, the missing value analysis and the preprocessing decision would be based on the training data set only. The same preprocessing rules will then be applied in the validation and the test data set.
The first step is to calculate the percentage of the missing values in each of the training features. And this will help to identify the features with the high level of the missing data and guide the selection of suitable preprocessing methods.


In [18]:

# Section 2.4: Missing Value Analysis (Training Data Only)

# Exclude the target and identifier/time columns
excluded_cols = ["isFraud", "TransactionID", "TransactionDT"]

feature_cols = [
    col for col in train_df.columns
    if col not in excluded_cols
]

# Calculate missing-value percentages in training features
missing_percentage = (
    train_df[feature_cols]
    .isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print("MISSING VALUE ANALYSIS - TRAINING DATA")
print("-" * 50)

print("Number of features:", len(feature_cols))

print(
    "Features with missing values:",
    (missing_percentage > 0).sum()
)

print(
    "Features with more than 90% missing values:",
    (missing_percentage > 90).sum()
)

print("\nTop 20 features with highest missing percentages:")

display(
    missing_percentage.head(20).to_frame(
        name="Missing Percentage (%)"
    ).round(2)
)


MISSING VALUE ANALYSIS - TRAINING DATA
--------------------------------------------------
Number of features: 391
Features with missing values: 331
Features with more than 90% missing values: 2

Top 20 features with highest missing percentages:


,Missing Percentage (%)
D7,93.53
dist2,92.95
D13,89.34
D14,89.12
D12,88.56
D6,87.28
D8,86.41
D9,86.41
V139,84.23
V150,84.23


### 2.4.1 The removing of the futures with the extremely high missing values.

Features with more than 90% missing values in the training data set are removed for the initial modeling experiments. This threshold is selected to reduce the number of the highly incomplete features and simplify the preprocessing.
The decision to remove the removal is based only on training data to prevent the data leakage the same features are then removed from the validation and the testing data set to maintain the consistent feature structure. And the effect of this decision that may be examined in the experiments later is that the missing value patterns can also contain useful information for fraud detection.


In [19]:

# Section 2.4.1: Remove extremely sparse features

# Identify features with more than 90% missing values
high_missing_cols = missing_percentage[
    missing_percentage > 90
].index.tolist()

print("Features to remove:", high_missing_cols)

# Remove the same columns from all three datasets
train_df = train_df.drop(columns=high_missing_cols)
val_df = val_df.drop(columns=high_missing_cols)
test_df = test_df.drop(columns=high_missing_cols)

print("\nAFTER REMOVING HIGH-MISSING FEATURES")
print("-" * 45)

print("Training set:", train_df.shape)
print("Validation set:", val_df.shape)
print("Testing set:", test_df.shape)

# Confirm the datasets have matching columns
assert train_df.columns.equals(val_df.columns)
assert train_df.columns.equals(test_df.columns)

print("\nAll datasets have matching columns!")


Features to remove: ['D7', 'dist2']

AFTER REMOVING HIGH-MISSING FEATURES
---------------------------------------------
Training set: (413378, 392)
Validation set: (88581, 392)
Testing set: (88581, 392)

All datasets have matching columns!



### 2.4.2 Handling the remaining missing values

After removing the features with the more than 90% missing values, the remaining of the missing values are handled using the suitable imputation technique.

Numerical features will then be filled using the median values calculated from the training data set, while the missing categorical values will be presented by the separate missing category. The numerically missing values indicators will also be considered to preserve the information about the original missing values.

The preprocessing parameters they unlearned from the training data only and applied consistently to the validation and the testing datasets. And this helps to prevent data leakage and to support a fair comparison between machine learning and the deep learning models.


In [20]:

# Section 2.4.2: Identify of the numerical and categorical features
# Identify of the feature types

# Exclude target, transaction identifier and time
excluded_cols = ["isFraud", "TransactionID", "TransactionDT"]

feature_cols = [
    col for col in train_df.columns
    if col not in excluded_cols
]

# Identify numerical features
numerical_features = train_df[feature_cols].select_dtypes(
    include=["number"]
).columns.tolist()

# Identify categorical features
categorical_features = train_df[feature_cols].select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("FEATURE TYPES AFTER REMOVING SPARSE COLUMNS")
print("-" * 50)

print("Total modelling features:", len(feature_cols))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nNumerical features with missing values:",
      train_df[numerical_features].isna().any().sum())

print("Categorical features with missing values:",
      train_df[categorical_features].isna().any().sum())


FEATURE TYPES AFTER REMOVING SPARSE COLUMNS
--------------------------------------------------
Total modelling features: 389
Numerical features: 375
Categorical features: 14

Numerical features with missing values: 316
Categorical features with missing values: 13


In [21]:

# Section 2.4.2: Handle remaining missing values

# Calculate numerical medians using training data only
train_medians = train_df[numerical_features].median()

# Some columns may contain only missing values in training
# Use 0 as a fallback for those columns
train_medians = train_medians.fillna(0)

# Fill numerical missing values in all three datasets
for data in [train_df, val_df, test_df]:
    data[numerical_features] = (
        data[numerical_features].fillna(train_medians)
    )

# Fill categorical missing values
for data in [train_df, val_df, test_df]:
    data[categorical_features] = (
        data[categorical_features]
        .fillna("Missing")
    )

print("MISSING VALUE IMPUTATION COMPLETED")
print("-" * 45)

print("Training missing values:",
      train_df[feature_cols].isna().sum().sum())

print("Validation missing values:",
      val_df[feature_cols].isna().sum().sum())

print("Testing missing values:",
      test_df[feature_cols].isna().sum().sum())

print("\nTraining shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Testing shape:", test_df.shape)


MISSING VALUE IMPUTATION COMPLETED
---------------------------------------------
Training missing values: 0
Validation missing values: 0
Testing missing values: 0

Training shape: (413378, 392)
Validation shape: (88581, 392)
Testing shape: (88581, 392)


### 2.5 Encoding the Categorical Features

The dataset contains categorical features, and it must be converted into numerical representations before the model training. Ordinal encoding it is used as an initial approach to assign the numerical identifier to each of the categories.

The encoder is fitted using the training of the dataset only and then also applied to validation and the testing datasets. Categories not observed during the training are assigned the separate unknown category code.

The encoded of the values represent category identifiers rather than the actual order. The Model specific categories representations, which are such as embeddings, and may be used in the later deep learning experience.


In [22]:

# Section 2.5: Encoding Categorical Features

from sklearn.preprocessing import OrdinalEncoder

# Create an encoder that can handle new categories
encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.float32
)

# Convert categorical features to strings
# and fit the encoder using training data only
train_categories = train_df[categorical_features].astype(str)

encoder.fit(train_categories)

# Apply the same encoder to all datasets
for data in [train_df, val_df, test_df]:
    data[categorical_features] = encoder.transform(
        data[categorical_features].astype(str)
    )

print("CATEGORICAL ENCODING COMPLETED")
print("-" * 45)

print("Number of encoded features:", len(categorical_features))

print("\nTraining shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Testing shape:", test_df.shape)

# Check whether any text columns remain in model features
remaining_text_cols = train_df[feature_cols].select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nRemaining text features:", remaining_text_cols)

print("\nExample encoded categorical features:")
display(train_df[categorical_features].head())


CATEGORICAL ENCODING COMPLETED
---------------------------------------------
Number of encoded features: 14

Training shape: (413378, 392)
Validation shape: (88581, 392)
Testing shape: (88581, 392)

Remaining text features: []

Example encoded categorical features:


,ProductCD,card4,card6,P_emaildomain,R_emaildomain,M1,M2,M3,M4,M5,M6,M7,M8,M9
0,4.0,2.0,2.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,2.0,1.0,1.0,1.0
1,4.0,3.0,2.0,17.0,0.0,1.0,1.0,1.0,0.0,2.0,2.0,1.0,1.0,1.0
2,4.0,4.0,3.0,36.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4.0,3.0,3.0,54.0,0.0,1.0,1.0,1.0,0.0,2.0,0.0,1.0,1.0,1.0
4,1.0,3.0,2.0,17.0,0.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0


### 2.6 Preparing the Model inputs and the Targets

After the handling of the missing values and encoding categorical features, dataset it is separated into the features (X) and the target labels (y). The target variable isFraud, it indicated whether the transaction is fraudulent.

The TransactionID and the TransactionDT columns are excluded from the inputs of the model. Transaction time is preserved separately for the evaluating performance on the later transactions.

The same feature columns are used for training, the validation and the testing set to support the consistent comparison of the model. The testing labels would be reserved for the final evaluation and will not be used for the training model or the hyperparameter tuning.


In [23]:

# Section 2.6: Prepare model inputs and targets

# Columns excluded from model inputs
excluded_cols = ["isFraud", "TransactionID", "TransactionDT"]

# Select the same features for all three datasets
feature_cols = [
    col for col in train_df.columns
    if col not in excluded_cols
]

# Prepare input features
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

# Prepare target labels
y_train = train_df["isFraud"]
y_val = val_df["isFraud"]
y_test = test_df["isFraud"]

# Preserve transaction time for temporal evaluation
time_train = train_df["TransactionDT"]
time_val = val_df["TransactionDT"]
time_test = test_df["TransactionDT"]

print("MODEL INPUTS AND TARGETS")
print("-" * 45)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("\nNumber of modelling features:", len(feature_cols))

# Verify that the datasets contain no missing model inputs
assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

# Verify matching feature columns
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

print("\nModel inputs verified successfully!")


MODEL INPUTS AND TARGETS
---------------------------------------------
X_train: (413378, 389)
X_val: (88581, 389)
X_test: (88581, 389)

y_train: (413378,)
y_val: (88581,)
y_test: (88581,)

Number of modelling features: 389

Model inputs verified successfully!


### 2.7 The final validation and the reproducibility

The preprocessed data set was successfully prepared and verified in part 2.6. And on this stage Performance also check on the target labels and the chronologically ordered to confirm that the data set are suitable for the fraud detection experiments.

The original data set is accessed from the Kaggle using the Kaggle hub. To support the reproducibility, and the preprocessing produces will then be organized into the resemble quote and also included in the GitHub repository, allowing the same transformation to be applied without depending on the personal Google Drive storage.


In [24]:

# Section 2.7: Final Validation and Reproducibility

print("FINAL VALIDATION")
print("-" * 45)

# Verify valid fraud labels
for name, y in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    assert not y.isna().any()
    assert set(y.unique()).issubset({0, 1})
    print(f"{name} labels verified.")

# Verify chronological order within each dataset
assert time_train.is_monotonic_increasing
assert time_val.is_monotonic_increasing
assert time_test.is_monotonic_increasing

# Verify chronological order between datasets
assert time_train.max() <= time_val.min()
assert time_val.max() <= time_test.min()

print("\nChronological order verified!")

# Verify that all model features are numerical
assert all(
    pd.api.types.is_numeric_dtype(dtype)
    for dtype in X_train.dtypes
)

print("All model features are numerical.")

print("\nFinal validation completed successfully!")


FINAL VALIDATION
---------------------------------------------
Training labels verified.
Validation labels verified.
Testing labels verified.

Chronological order verified!
All model features are numerical.

Final validation completed successfully!


### 2.8 A summary of preprocessing

This notebook prepared the IEEE-CIS fraud detection transaction data set for the machine learning and deep learning experiments, and the data set initially contained 590,540 transactions and also 394 columns.

The transaction was divided chronologically into the training (70%), and the validation (15%) as well as the testing (15%) sets. Two features, D7 and dist2, were removed because they contained more than 90% missing values in the training sets. And the remaining numerical missing values were replaced using the training set Medians where the category missing fellows it was assigned a separate missing category. The 14 category features were then converted into numerical identifiers by using ordinary encoding fitted on the training set.

The final data set contained 413,378 training transitions 88,581 validation transaction as well as 88,581 testing transactions, and each with 389 modeling features. The final chucks confirmed the consistent feature structures valid fraud and no missing models failed as well as chronological ordering.

The prepared data set will then support the comparison between the gradient boosted tree model and the learning architecture as well as the experiments on the glass imbalance and the performance over time. And the preprocessing steps will then be made reusable in the project repository to support reproducibility.
